# FinScope AI — Model Training & Evaluation Notebook

This notebook walks through the complete training pipeline for the FinScope AI credit risk prediction system:

1. **Data Generation & Exploration** — Generate synthetic financial data and perform EDA
2. **Feature Engineering** — Build engineered features, handle missing values and outliers
3. **Model Training** — Train Logistic Regression (baseline), XGBoost, and PyTorch DNN
4. **Evaluation** — Compare models using ROC-AUC, Precision-Recall, Confusion Matrix, Calibration Curve
5. **Explainability** — SHAP-based feature importance (global & local)
6. **Artifact Export** — Save all models and pipeline for production deployment

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, roc_auc_score

from data.generate_data import generate_synthetic_data
from data.feature_engineering import (
    FeaturePipeline, engineer_features, handle_missing_values,
    handle_outliers, split_data, FEATURE_COLUMNS, TARGET_COLUMN,
)
from models.logistic_model import LogisticRegressionModel
from models.xgboost_model import XGBoostModel
from models.dnn_model import DNNModel
from models.evaluation import compute_metrics, plot_all_metrics
from models.explainability import SHAPExplainer

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

ARTIFACTS_DIR = PROJECT_ROOT / "models" / "artifacts"
PLOTS_DIR = PROJECT_ROOT / "models" / "plots"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete ✔")

## 1. Data Generation & Exploration

In [ ]:
# Generate synthetic financial data
df = generate_synthetic_data(n_samples=10_000, random_seed=42)

print(f"Dataset shape: {df.shape}")
print(f"\nDefault rate:\n{df['default'].value_counts(normalize=True)}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default distribution
df["default"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Default Distribution", fontsize=14)
axes[0].set_xticklabels(["No Default (0)", "Default (1)"], rotation=0)
axes[0].set_ylabel("Count")

# Correlation heatmap (top features)
key_cols = [
    "annual_income", "total_debt", "credit_limit", "credit_score",
    "num_missed_payments_last_12m", "loan_amount", "interest_rate",
    "savings_balance", "has_bankruptcy", "default"
]
corr = df[key_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1],
            square=True, linewidths=0.5)
axes[1].set_title("Feature Correlation Matrix", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by default status
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_features = ["annual_income", "credit_score", "total_debt",
                 "num_missed_payments_last_12m", "interest_rate", "loan_amount"]

for ax, feat in zip(axes.flat, plot_features):
    for label, color in [(0, "steelblue"), (1, "coral")]:
        subset = df[df["default"] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.6, color=color,
                label=f"{'Default' if label else 'No Default'}")
    ax.set_title(feat, fontsize=12)
    ax.legend()

plt.suptitle("Feature Distributions by Default Status", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Feature Engineering Pipeline

In [ ]:
# Split data with stratification
train_df, val_df, test_df = split_data(df, test_size=0.15, val_size=0.15, random_state=42)

print(f"Train: {train_df.shape} | Default rate: {train_df['default'].mean():.3f}")
print(f"Val:   {val_df.shape} | Default rate: {val_df['default'].mean():.3f}")
print(f"Test:  {test_df.shape} | Default rate: {test_df['default'].mean():.3f}")

In [ ]:
# Fit feature pipeline on training data
pipeline = FeaturePipeline()
X_train, y_train = pipeline.fit_transform(train_df)

# Transform validation and test
X_val = pipeline.transform(val_df)
y_val = val_df[TARGET_COLUMN].values.astype(np.float32)

X_test = pipeline.transform(test_df)
y_test = test_df[TARGET_COLUMN].values.astype(np.float32)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Features ({len(pipeline.feature_columns)}): {pipeline.feature_columns[:10]}...")

## 3. Model Training

### 3.1 Logistic Regression (Baseline)

In [ ]:
lr_model = LogisticRegressionModel(C=0.5)
lr_metrics = lr_model.train(X_train, y_train, X_val, y_val)

test_proba_lr = lr_model.predict_proba(X_test)
lr_test = compute_metrics(y_test, test_proba_lr)

print("\n--- Logistic Regression Results ---")
for k, v in lr_test.items():
    print(f"  {k}: {v:.4f}")

### 3.2 XGBoost

In [ ]:
xgb_model = XGBoostModel()
xgb_metrics = xgb_model.train(X_train, y_train, X_val, y_val, tune_hyperparams=False)

test_proba_xgb = xgb_model.predict_proba(X_test)
xgb_test = compute_metrics(y_test, test_proba_xgb)

print("\n--- XGBoost Results ---")
for k, v in xgb_test.items():
    print(f"  {k}: {v:.4f}")

# Feature importance
importance = xgb_model.get_feature_importance()
feat_imp = pd.Series(
    {pipeline.feature_columns[i]: v for i, v in importance.items()}
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot.barh(ax=ax, color="steelblue")
ax.set_title("XGBoost Feature Importance (Top 15)", fontsize=14)
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

### 3.3 Deep Neural Network (PyTorch)

In [ ]:
dnn_model = DNNModel(
    hidden_dims=(128, 64, 32),
    dropout=0.3,
    learning_rate=1e-3,
    batch_size=256,
    epochs=100,
    patience=10,
)
dnn_metrics = dnn_model.train(X_train, y_train, X_val, y_val)

test_proba_dnn = dnn_model.predict_proba(X_test)
dnn_test = compute_metrics(y_test, test_proba_dnn)

print("\n--- DNN Results ---")
for k, v in dnn_test.items():
    print(f"  {k}: {v:.4f}")

## 4. Model Comparison & Evaluation

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "DNN (PyTorch)"],
    "ROC-AUC": [lr_test["roc_auc"], xgb_test["roc_auc"], dnn_test["roc_auc"]],
    "Avg Precision": [lr_test["average_precision"], xgb_test["average_precision"], dnn_test["average_precision"]],
    "Accuracy": [lr_test["accuracy"], xgb_test["accuracy"], dnn_test["accuracy"]],
    "Precision": [lr_test["precision"], xgb_test["precision"], dnn_test["precision"]],
    "Recall": [lr_test["recall"], xgb_test["recall"], dnn_test["recall"]],
    "F1 Score": [lr_test["f1_score"], xgb_test["f1_score"], dnn_test["f1_score"]],
})

comparison = comparison.round(4)
print("\n" + "=" * 80)
print("MODEL COMPARISON ON TEST SET")
print("=" * 80)
display(comparison)

best_model = comparison.loc[comparison["ROC-AUC"].idxmax(), "Model"]
print(f"\n🏆 Best model by ROC-AUC: {best_model}")

In [ ]:
# ROC Curves — All Models Overlaid
fig, ax = plt.subplots(figsize=(9, 7))

for name, proba, color in [
    ("Logistic Regression", test_proba_lr, "#4CAF50"),
    ("XGBoost", test_proba_xgb, "#2196F3"),
    ("DNN (PyTorch)", test_proba_dnn, "#FF5722"),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {roc_auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Model Comparison", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / "roc_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

# Confusion Matrices Side-by-Side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, proba) in zip(axes, [
    ("Logistic Regression", test_proba_lr),
    ("XGBoost", test_proba_xgb),
    ("DNN", test_proba_dnn),
]):
    cm = confusion_matrix(y_test, (proba >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Default", "Default"],
                yticklabels=["No Default", "Default"])
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

# Calibration Curves
fig, ax = plt.subplots(figsize=(9, 7))

for name, proba, color in [
    ("Logistic Regression", test_proba_lr, "#4CAF50"),
    ("XGBoost", test_proba_xgb, "#2196F3"),
    ("DNN", test_proba_dnn, "#FF5722"),
]:
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10)
    ax.plot(prob_pred, prob_true, marker="o", lw=2, color=color, label=name)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability", fontsize=12)
ax.set_ylabel("Fraction of Positives", fontsize=12)
ax.set_title("Calibration Curves", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / "calibration_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. SHAP Explainability

Using SHAP (SHapley Additive exPlanations) to understand feature contributions to predictions.

In [ ]:
# Initialize SHAP explainer with XGBoost model
explainer = SHAPExplainer(
    model=xgb_model.model,
    feature_names=pipeline.feature_columns,
    model_type="tree",
)

# Global feature importance
global_importance = explainer.global_importance(X_test[:500], top_k=15)

print("\nGlobal Feature Importance (mean |SHAP|):")
for feat, val in global_importance.items():
    print(f"  {feat:40s} {val:.4f}")

In [ ]:
# SHAP Summary Plot
_ = explainer.plot_summary(
    X_test[:500],
    save_path=str(PLOTS_DIR / "shap_summary.png"),
    max_display=15,
)
plt.show()

In [ ]:
# Local explanation for a single prediction
sample_idx = 0
sample = X_test[sample_idx:sample_idx+1]
sample_prob = xgb_model.predict_proba(sample)[0]

print(f"Sample #{sample_idx} — Predicted default probability: {sample_prob:.4f}")

local_exp = explainer.local_explanation(sample)
print("\nTop contributing features:")
sorted_exp = sorted(local_exp.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
for feat, val in sorted_exp:
    direction = "+" if val > 0 else "-"
    print(f"  {direction} {feat:40s} {val:+.4f}")

# Waterfall plot
_ = explainer.plot_waterfall(
    sample,
    save_path=str(PLOTS_DIR / "shap_waterfall_sample.png"),
)
plt.show()

## 6. Save Artifacts

In [ ]:
# Save all models and pipeline
pipeline.save(str(ARTIFACTS_DIR))
lr_model.save(str(ARTIFACTS_DIR))
xgb_model.save(str(ARTIFACTS_DIR))
dnn_model.save(str(ARTIFACTS_DIR))

# Save training metrics
all_metrics = {
    "logistic_regression": lr_test,
    "xgboost": xgb_test,
    "dnn": dnn_test,
}
with open(ARTIFACTS_DIR / "training_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

# Save feature names
with open(ARTIFACTS_DIR / "feature_names.json", "w") as f:
    json.dump(pipeline.feature_columns, f, indent=2)

# Summary
print("=" * 60)
print("ARTIFACTS SAVED")
print("=" * 60)
for p in sorted(ARTIFACTS_DIR.glob("*")):
    size = p.stat().st_size / 1024
    print(f"  {p.name:40s} {size:8.1f} KB")

print(f"\n\nPlots saved to: {PLOTS_DIR}")
for p in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {p.name}")

print("\nTraining complete! Models are ready for deployment.")